# Swing detector sanity check

Purpose:
- replay saved candles through the detector objects bar by bar
- compare confirmed detector events against stored fractal and rule labels
- inspect a quick visual slice of the detected swings

Inputs:
- reduced features parquet
- fractal labels parquet
- rule labels parquet
- saved model and threshold artifacts referenced by the detector classes

Reading guide:
1. Normalize indexes and merge all datasets on `timestamp`.
2. Recover the OHLC columns needed for replay.
3. Run the detectors in strict chronological order.
4. Compare confirmed events with label columns using precision, recall, and F1.
5. Plot a short window for a fast visual sanity check.

Interpretation note:
- The detector emits confirmed events, not immediate labels at the current bar, so sparse-event rates matter as much as the raw classification scores.


In [ ]:
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def locate_ml_root(start=None) -> Path:
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start] + list(start.parents):
        if candidate.name == "ML v1" and (candidate / "data").exists():
            return candidate

        ml_root = candidate / "ML v1"
        if (ml_root / "data").exists() and (ml_root / "code").exists():
            return ml_root

    raise FileNotFoundError(
        "Could not locate the 'ML v1' workspace from the current working directory."
    )


ML_ROOT = locate_ml_root()
PROJECT_ROOT = ML_ROOT.parent
DATA_DIR = ML_ROOT / "data"
CONFIG_DIR = ML_ROOT / "configs"
CODE_DIR = ML_ROOT / "code"

FEATURES_REDUCED_PATH = DATA_DIR / "ETHUSDT_15m_features_reduced.parquet"
FRACTAL_LABELS_PATH = DATA_DIR / "ETHUSDT_15m_fractal_labels_L10_R10.parquet"
RULE_LABELS_PATH = DATA_DIR / "swing_labels.parquet"
OHLC_PATH = DATA_DIR / "ETHUSDT_15m_ohlc_clean.parquet"
MODEL_HIGH_PATH = DATA_DIR / "model_high.pkl"
MODEL_LOW_PATH = DATA_DIR / "model_low.pkl"
MODEL_RULE_HIGH_PATH = DATA_DIR / "model_rule_high_baseline.pkl"
MODEL_RULE_LOW_PATH = DATA_DIR / "model_rule_low_baseline.pkl"
FEATURE_CONFIG_PATH = CONFIG_DIR / "feature_config.json"
THRESHOLDS_PATH = CONFIG_DIR / "thresholds.json"

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from swing_detector_ml_fractal import SwingDetectorMLFractal

try:
    from swing_detector_ml_rule_v2 import SwingDetectorMLRule
except ImportError:
    SwingDetectorMLRule = None


In [ ]:
def set_timestamp_index(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    if "timestamp" not in out.columns:
        raise ValueError(f"{name}: no 'timestamp' column. Columns: {list(out.columns)[:20]}")
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce")
    out = out.dropna(subset=["timestamp"]).set_index("timestamp").sort_index()
    out = out[~out.index.duplicated(keep="last")]
    return out

def normalize_dt_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.to_datetime(out.index, utc=True, errors="coerce")
    out = out[~out.index.isna()].sort_index()
    out = out[~out.index.duplicated(keep="last")]
    return out


In [ ]:
# Merge the three data sources on the same timeline so detector output and reference labels can be compared bar by bar.

# --- Load ---
features = pd.read_parquet(FEATURES_REDUCED_PATH)
fractal_raw = pd.read_parquet(FRACTAL_LABELS_PATH)
rule_raw = pd.read_parquet(RULE_LABELS_PATH)

# features index is already datetime in your pipeline, but normalize anyway
features = normalize_dt_index(features)

# labels: timestamp is a column -> move to index
fractal = set_timestamp_index(fractal_raw, "fractal")
rule = set_timestamp_index(rule_raw, "rule")

# rename targets to explicit names
fractal = fractal.rename(columns={"y_high": "y_high_fractal", "y_low": "y_low_fractal"})
rule = rule.rename(columns={"y_high": "y_high_rule", "y_low": "y_low_rule"})

# drop meta cols if present
drop_cols = ["segment_id"]
fractal = fractal.drop(columns=[c for c in drop_cols if c in fractal.columns], errors="ignore")
rule = rule.drop(columns=[c for c in drop_cols if c in rule.columns], errors="ignore")

# merge
df = features.join(fractal, how="inner").join(rule, how="inner")
print("df shape:", df.shape)
df.head()


In [ ]:
# If your features parquet already contains OHLC, this will work.
# Otherwise, you need to join OHLC from a candles parquet first.

def find_ohlc_cols(columns):
    low = {c.lower(): c for c in columns}
    candidates = [
        ("open", "high", "low", "close"),
        ("o", "h", "l", "c"),
        ("Open", "High", "Low", "Close"),
    ]
    for a,b,c,d in candidates:
        if a.lower() in low and b.lower() in low and c.lower() in low and d.lower() in low:
            return [low[a.lower()], low[b.lower()], low[c.lower()], low[d.lower()]]
    return None

ohlc_cols = find_ohlc_cols(df.columns)
print("OHLC cols:", ohlc_cols)


In [ ]:
# If OHLC is missing, you must load it from your candles source and join by timestamp.
# Example:
# candles = pd.read_parquet(OHLC_PATH)
# candles = set_timestamp_index(candles, "candles") if "timestamp" in candles.columns else normalize_dt_index(candles)
# candles = candles.rename(columns=str.lower)
# candles = candles[["open","high","low","close"]]
# df = df.join(candles, how="inner")
# ohlc_cols = ["open","high","low","close"]


In [ ]:
# These detector instances are configured from saved training artifacts so this notebook can replay the online contract.

# Instantiate detectors
det_fr = SwingDetectorMLFractal(
    model_high_path=str(MODEL_HIGH_PATH),
    model_low_path=str(MODEL_LOW_PATH),
    feature_config_path=str(FEATURE_CONFIG_PATH),
    thresholds_path=str(THRESHOLDS_PATH),
)

# Uncomment when you have working rule artifacts and lightgbm deps
# det_rl = SwingDetectorMLRule(
#     model_high_path=str(MODEL_RULE_HIGH_PATH),
#     model_low_path=str(MODEL_RULE_LOW_PATH),
#     feature_config_path=str(FEATURE_CONFIG_PATH),
#     threshold_high=0.5,
#     threshold_low=0.5,
#     right=0,
# )


In [ ]:
# Replay one bar at a time and collect only the events that leave the detector confirmation queue.

def run_detector(det, df_ohlc: pd.DataFrame) -> pd.DataFrame:
    confirmed = []
    for ts, row in df_ohlc.iterrows():
        bar = SimpleNamespace(open=float(row.open), high=float(row.high), low=float(row.low), close=float(row.close))
        det.update(bar)
        confirmed.extend(det.pop_confirmed())

    rows = []
    for s in confirmed:
        ts_swing = df_ohlc.index[int(s.index)]
        rows.append({
            "timestamp": ts_swing,
            "is_high": 1 if s.highlow == 1 else 0,
            "is_low": 1 if s.highlow == -1 else 0,
            "p_high": float(s.p_high),
            "p_low": float(s.p_low),
            "level": float(s.level),
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.set_index("timestamp").sort_index()


In [ ]:
ohlc = df[ohlc_cols].copy()
ohlc.columns = ["open","high","low","close"]

out_fr = run_detector(det_fr, ohlc)
print("Confirmed fractal swings:", len(out_fr))
out_fr.head()


In [ ]:
# Metrics are computed on sparse event indicators after joining detector output back to the reference label timeline.

from sklearn.metrics import precision_recall_fscore_support

def eval_against(y_high_col: str, y_low_col: str, det_out: pd.DataFrame, name: str):
    y = df[[y_high_col, y_low_col]].copy()
    y.columns = ["y_high", "y_low"]

    if det_out.empty:
        merged = y.copy()
        merged["is_high"] = 0
        merged["is_low"] = 0
    else:
        merged = y.join(det_out[["is_high","is_low"]], how="left").fillna(0)

    ph = precision_recall_fscore_support(merged["y_high"], merged["is_high"], average="binary", zero_division=0)
    pl = precision_recall_fscore_support(merged["y_low"], merged["is_low"], average="binary", zero_division=0)

    print(f"[{name}] High: precision={ph[0]:.3f} recall={ph[1]:.3f} f1={ph[2]:.3f} | rate={merged.is_high.mean():.4f} true={merged.y_high.mean():.4f}")
    print(f"[{name}] Low : precision={pl[0]:.3f} recall={pl[1]:.3f} f1={pl[2]:.3f} | rate={merged.is_low.mean():.4f}  true={merged.y_low.mean():.4f}")

eval_against("y_high_fractal", "y_low_fractal", out_fr, "ML-Fractal vs Fractal labels")


In [ ]:
# This plot is intentionally lightweight: it is only meant to catch obvious timestamp or confirmation-shift mistakes.

# Quick visual sanity on a slice
start = 5000
end = start + 600
sl = df.iloc[start:end].copy()

fig = plt.figure(figsize=(12,4))
plt.plot(sl["close"].values)

if not out_fr.empty:
    fr_idx = out_fr.index[(out_fr.index >= sl.index[0]) & (out_fr.index <= sl.index[-1])]
    x = sl.index.get_indexer(fr_idx)
    plt.scatter(x, sl.loc[fr_idx, "close"].values, marker="^")

plt.title("Close with confirmed swings (^) fractal ML")
plt.show()
